# The Training Loop: Model Engine Room

Reach for this when you need: 
- Standard boilerplate for training, validation, and early stopping.
- Reference for saving and loading full model checkpoints.
- Logic for switching between `model.train()` and `model.eval()`.

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm  # nice progress bar while training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 0. Minimal Dummy Setup (run this first!)

All code blocks below reference `model`, `optimizer`, `loader`, and `criterion`.  
This cell defines minimal stand-ins so every block in this notebook is independently executable.

In [7]:
# Dummy model: 2-layer MLP (input=10, hidden=32, output=1)
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
).to(device)

# Loss function & optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Dummy DataLoader (random regression data)
from torch.utils.data import DataLoader, TensorDataset

N = 200
X_dummy = torch.randn(N, 10)
y_dummy = torch.randn(N, 1)

dataset = TensorDataset(X_dummy, y_dummy)
loader  = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Model: {model}")
print(f"Batches per epoch: {len(loader)}")

Model: Sequential(
  (0): Linear(in_features=10, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=1, bias=True)
)
Batches per epoch: 7


## 1. Standard Boilerplate Loop

| State | Description | Purpose |
| :--- | :--- | :--- |
| `model.train()` | Enables Dropout and BatchNorm update | Ensures weights learn |
| `model.eval()` | Disables Dropout, freezes BatchNorm | Consistent predictions for validation |
| `torch.no_grad()` | Disables gradient graph construction | Memory and compute optimization during inference |

In [8]:
def train_epoch(model, loader, optimizer, criterion):
    """Run one full pass over the training data."""
    model.train()          # activates dropout / batchnorm training mode
    total_loss = 0
    for x, y in tqdm(loader, desc="train", leave=False):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()            # 1. clear old gradients
        outputs = model(x)               # 2. forward pass
        loss    = criterion(outputs, y)  # 3. compute loss
        loss.backward()                  # 4. backprop
        optimizer.step()                 # 5. update weights

        total_loss += loss.item()        # .item() detaches from graph -- avoids memory leak
    return total_loss / len(loader)      # average loss per batch


def validate(model, loader, criterion):
    """Evaluate the model on a validation set -- no gradients needed."""
    model.eval()           # disables dropout / freezes batchnorm stats
    total_loss = 0
    with torch.no_grad():  # turns off autograd engine for speed + memory
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            loss    = criterion(outputs, y)
            total_loss += loss.item()
    return total_loss / len(loader)


# Quick smoke test
train_loss = train_epoch(model, loader, optimizer, criterion)
val_loss   = validate(model, loader, criterion)
print(f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")

Train loss: 1.0285 | Val loss: 1.0235


## 2. Full Training Loop with Early Stopping

In [9]:
NUM_EPOCHS        = 10
PATIENCE          = 3    # stop if val loss doesn't improve for 3 epochs

best_val_loss     = float('inf')
epochs_no_improve = 0
history           = {'train': [], 'val': []}

for epoch in range(1, NUM_EPOCHS + 1):
    t_loss = train_epoch(model, loader, optimizer, criterion)
    v_loss = validate(model, loader, criterion)

    history['train'].append(t_loss)
    history['val'].append(v_loss)

    print(f"Epoch {epoch:02d} | train: {t_loss:.4f} | val: {v_loss:.4f}")

    # early stopping check
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_model_weights.pth")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch}.")
            break

print(f"\nBest val loss: {best_val_loss:.4f}")

Epoch 01 | train: 1.0103 | val: 0.9624


Epoch 02 | train: 1.1076 | val: 1.0329


Epoch 03 | train: 1.1164 | val: 0.9589


Epoch 04 | train: 0.9124 | val: 0.8922


Epoch 05 | train: 0.9343 | val: 0.9633


Epoch 06 | train: 0.9800 | val: 0.9377


Epoch 07 | train: 0.8948 | val: 0.8847


Epoch 08 | train: 0.8805 | val: 0.8927


Epoch 09 | train: 0.8905 | val: 0.8977


Epoch 10 | train: 0.9898 | val: 0.9782
Early stopping triggered at epoch 10.

Best val loss: 0.8847


## 3. Checkpointing: Saving & Loading

**Always save the `state_dict`, not the model object.**

Use when: Capturing training progress every N epochs or when val loss hits a new minimum.  
Don't use when: Deploying for inference where you might want TorchScript/ONNX (though `state_dict` still works for both).

In [10]:
# Saving a full training checkpoint
checkpoint = {
    'epoch'               : 10,
    'model_state_dict'    : model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss'                : best_val_loss,
}

torch.save(checkpoint, "full_checkpoint.pth")
print("Checkpoint saved.")

# Loading and resuming training
checkpoint = torch.load("full_checkpoint.pth", map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch'] + 1

print(f"Resumed from epoch {checkpoint['epoch']}, loss={checkpoint['loss']:.4f}")

Checkpoint saved.
Resumed from epoch 10, loss=0.8847


## 4. Gradient Clipping

Prevents exploding gradients in RNNs / Transformers. Clip **before** `optimizer.step()`.

Use when: Training LSTMs, Transformers, or any deep residual network.  
Don't use when: Gradients are already well-behaved (wastes time).

In [11]:
MAX_GRAD_NORM = 1.0

model.train()
x, y = next(iter(loader))
x, y = x.to(device), y.to(device)

optimizer.zero_grad()
loss = criterion(model(x), y)
loss.backward()

# Clip BEFORE stepping -- norms all parameter gradients to max_norm
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)

optimizer.step()
print("Step with clipping done.")

Step with clipping done.


## 5. Learning Rate Scheduling

| Scheduler | Description | Best For |
| :--- | :--- | :--- |
| `StepLR` | Decay by gamma every N steps | Stable, predictable decay |
| `ReduceLROnPlateau` | Decay when metric stops improving | When you don't know the right schedule |
| `CosineAnnealingLR` | Smooth cosine wave decay | Vision / LLM pre-training |
| `OneCycleLR` | Warm-up + anneal in one cycle | Super-convergence, fast training |

In [12]:
# CosineAnnealingLR example
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

for epoch in range(1, 4):
    train_epoch(model, loader, optimizer, criterion)
    scheduler.step()  # call AFTER each epoch (not each batch for most schedulers)
    print(f"Epoch {epoch} | LR: {scheduler.get_last_lr()[0]:.6f}")

# ReduceLROnPlateau (metric-aware)
plateau_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)
val_loss = validate(model, loader, criterion)
plateau_scheduler.step(val_loss)  # pass the monitored metric

Epoch 1 | LR: 0.000976


Epoch 2 | LR: 0.000905


Epoch 3 | LR: 0.000794


c:\Users\kunjs\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


### Common Pitfalls
- **Evaluation mode**: Forgetting `model.eval()` keeps Dropout active, making val metrics unstable.
- **Item loss**: Always call `loss.item()` when accumulating metrics. Raw tensors carry the whole compute graph -> memory leak.
- **Optimizer state on resume**: MUST load `optimizer_state_dict` to restore momentum/running averages.
- **`map_location` on load**: Always pass `map_location=device` so a GPU checkpoint can load on CPU (and vice versa).
- **Scheduler vs optimizer step order**: Most schedulers call `scheduler.step()` **after** `optimizer.step()`.

### Key Takeaways
- Separate `train_epoch` / `validate` functions for clean, modular code.
- Checkpointing is essential for long jobs -- include epoch, loss, and both state dicts.
- `tqdm` provides vital progress visibility without touching actual training logic.
- Early stopping + LR scheduling together are the two most impactful regularization tools beyond model architecture.